# Hand Gesture CNN Baseline (Notebook)

- 기존 노트북의 LeapGestRecog 데이터셋 분할 규칙(train: 00~05, val: 06~07, test: 08~09)을 그대로 사용합니다.
- EfficientNet 대신 기본 CNN 모델을 학습/평가합니다.
- 마지막 셀에서 분류 리포트, 혼동행렬, 학습곡선을 확인합니다.

In [ ]:
# Colab GPU 전용: 필수 패키지 설치
!pip -q install kaggle seaborn scikit-learn

import os
try:
    import google.colab  # noqa: F401
except ImportError:
    raise EnvironmentError('이 노트북은 Google Colab 전용입니다. Colab에서 실행해 주세요.')

print('Colab environment confirmed.')

In [ ]:
import os
import copy
import random
from dataclasses import dataclass
from typing import List, Tuple

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def seed_everything(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 비활성화되어 있습니다. Colab에서 Runtime > Change runtime type > GPU로 변경하세요.')

device = torch.device('cuda')
print(f'Device: {device}')
print('GPU Name:', torch.cuda.get_device_name(0))

In [ ]:
# [Colab GPU 전용] Kaggle API로 LeapGestRecog 다운로드
# 실행 방법:
# 1) 이 셀 실행 시 나타나는 파일 선택 창에서 kaggle.json 업로드
# 2) KAGGLE_DATASET slug 확인 후 그대로 실행
# 3) 자동 다운로드/압축해제 후 BASE_PATH 탐지

import os
import shutil
import zipfile
import subprocess
from pathlib import Path
from google.colab import files

KAGGLE_DATASET = 'gti-upm/leapgestrecog'   # 필요시 수정
KAGGLE_DOWNLOAD_DIR = '/content/kaggle_data'
UNZIP_TARGET = '/content'

print('kaggle.json 파일을 업로드하세요.')
uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('kaggle.json 업로드가 필요합니다.')

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.makedirs(KAGGLE_DOWNLOAD_DIR, exist_ok=True)
cmd = [
    'kaggle', 'datasets', 'download',
    '-d', KAGGLE_DATASET,
    '-p', KAGGLE_DOWNLOAD_DIR,
    '--force'
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Kaggle 다운로드 실패: KAGGLE_DATASET slug 또는 kaggle.json 권한을 확인하세요.')

zip_files = list(Path(KAGGLE_DOWNLOAD_DIR).glob('*.zip'))
if not zip_files:
    raise FileNotFoundError('다운로드된 zip 파일을 찾지 못했습니다.')

for zf in zip_files:
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(UNZIP_TARGET)
    print(f'Unzipped: {zf}')

BASE_PATH_CANDIDATES = [
    '/content/leapGestRecog',
    '/content/gesture/leapGestRecog',
]

DETECTED_BASE_PATH = None
for p in BASE_PATH_CANDIDATES:
    if os.path.isdir(p):
        DETECTED_BASE_PATH = p
        break

if DETECTED_BASE_PATH is None:
    raise FileNotFoundError('leapGestRecog 폴더를 찾지 못했습니다. 압축 해제 경로를 확인하세요.')

print('Detected BASE_PATH:', DETECTED_BASE_PATH)

In [ ]:
# 데이터셋 루트 경로 설정 (앞 셀에서 자동 탐지된 경로 사용)
if 'DETECTED_BASE_PATH' not in globals() or not DETECTED_BASE_PATH:
    raise RuntimeError('먼저 [Colab GPU 전용] Kaggle 다운로드 셀을 실행하세요.')

BASE_PATH = DETECTED_BASE_PATH

TRAIN_SUBJECTS = ['00', '01', '02', '03', '04', '05']
VAL_SUBJECTS = ['06', '07']
TEST_SUBJECTS = ['08', '09']

INPUT_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 50
LR = 0.1
NUM_WORKERS = 2
OUTPUT_DIR = './cnn_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class LeapGestRecogDataset(Dataset):
    CLASSES = sorted([
        '01_palm', '02_l', '03_fist', '04_fist_moved', '05_thumb',
        '06_index', '07_ok', '08_palm_moved', '09_c', '10_down'
    ])

    def __init__(self, base_path: str, subjects: List[str], transform=None):
        self.transform = transform
        self.classes = self.CLASSES
        self.class_to_idx = {c: i for i, c in enumerate(self.CLASSES)}
        self.samples: List[Tuple[str, int]] = []

        for subj in subjects:
            for cls in self.CLASSES:
                cls_path = os.path.join(base_path, subj, cls)
                if not os.path.isdir(cls_path):
                    continue
                for fname in os.listdir(cls_path):
                    path = os.path.join(cls_path, fname)
                    if os.path.isfile(path):
                        self.samples.append((path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomAffine(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

if not os.path.isdir(BASE_PATH):
    raise FileNotFoundError(
        'Dataset path not found.\n'
        f'현재 BASE_PATH: {BASE_PATH}\n\n'
        '해결 방법:\n'
        '1) 바로 위 [Colab 전용] 데이터셋 준비 셀을 먼저 실행\n'
        '2) Drive 경로(DRIVE_DATASET_PATH) 또는 ZIP_PATH를 실제 경로로 수정\n'
        '3) BASE_PATH_CANDIDATES에 실제 폴더 경로 추가\n'
    )

train_dataset = LeapGestRecogDataset(BASE_PATH, TRAIN_SUBJECTS, train_transform)
val_dataset = LeapGestRecogDataset(BASE_PATH, VAL_SUBJECTS, eval_transform)
test_dataset = LeapGestRecogDataset(BASE_PATH, TEST_SUBJECTS, eval_transform)

if len(train_dataset) == 0 or len(val_dataset) == 0 or len(test_dataset) == 0:
    raise RuntimeError('데이터셋 분할 중 비어 있는 split이 있습니다. 폴더 구조를 확인하세요.')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Input size: {INPUT_SIZE}x{INPUT_SIZE}')
print(f'Train subjects {TRAIN_SUBJECTS}: {len(train_dataset):,} images')
print(f'Val subjects   {VAL_SUBJECTS}: {len(val_dataset):,} images')
print(f'Test subjects  {TEST_SUBJECTS}: {len(test_dataset):,} images')
print(f'Classes: {train_dataset.classes}')

In [ ]:
class BasicGestureCNN(nn.Module):
    """`주제_3_CNN으로_이미지_분류.ipynb`의 기본 CNN 설계를 손 제스처 10클래스에 맞게 적용."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)

@dataclass
class EpochResult:
    loss: float
    acc: float

def run_one_epoch(model, loader, criterion, optimizer, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return EpochResult(loss=total_loss / total, acc=(100.0 * correct / total))

model = BasicGestureCNN(num_classes=len(train_dataset.classes)).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'파라미터 수: {total_params:,}')

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-4
)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

print('Epoch | Train Loss | Train Acc | Val Loss | Val Acc | LR')
print('-' * 62)
for epoch in range(1, EPOCHS + 1):
    tr = run_one_epoch(model, train_loader, criterion, optimizer, device, train=True)
    va = run_one_epoch(model, val_loader, criterion, optimizer, device, train=False)
    scheduler.step()

    history['train_loss'].append(tr.loss)
    history['train_acc'].append(tr.acc)
    history['val_loss'].append(va.loss)
    history['val_acc'].append(va.acc)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'{epoch:>5} | {tr.loss:>10.4f} | {tr.acc:>8.2f}% | {va.loss:>8.4f} | {va.acc:>7.2f}% | {lr_now:.2e}')

    if va.acc > best_val_acc:
        best_val_acc = va.acc
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
model_path = os.path.join(OUTPUT_DIR, 'best_cnn_baseline.pt')
torch.save(model.state_dict(), model_path)
print(f'\nBest val acc: {best_val_acc:.2f}%')
print(f'Best model saved to: {model_path}')

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

class_names = train_dataset.classes
print('[Classification Report]')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - CNN Baseline')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'cnn_confusion_matrix.png')
plt.savefig(cm_path, dpi=200)
plt.show()
print(f'Confusion matrix saved to: {cm_path}')

plt.figure(figsize=(10, 4))
x = range(1, len(history['train_loss']) + 1)

plt.subplot(1, 2, 1)
plt.plot(x, history['train_loss'], label='Train Loss')
plt.plot(x, history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(x, history['train_acc'], label='Train Acc')
plt.plot(x, history['val_acc'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Curve')
plt.legend()

plt.tight_layout()
curve_path = os.path.join(OUTPUT_DIR, 'cnn_learning_curve.png')
plt.savefig(curve_path, dpi=200)
plt.show()
print(f'Learning curve saved to: {curve_path}')